# Immobilier et performance énergétique en France

Projet du cours Python pour la data science, ENSAE Paris.
Marc Olivier Bouiti Banza, Karlane Vasthi Foudjin Sipeuwo, Aniela Nomwendé Tania Zoungrana.

Ce notebook est le rapport du projet. Il s'exécute du début à la fin à partir des fichiers produits par la chaîne de traitement (voir le README).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE))
from src import chargement, config

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.2f}".format)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})

ORDRE = chargement.ETIQUETTES
COULEURS = {"A": "#009C6D", "B": "#52B153", "C": "#78BD76", "D": "#F4E70F",
            "E": "#F0B40F", "F": "#EB8235", "G": "#D7221F"}

## Introduction

*À rédiger.* La question, le contexte (réforme du DPE de juillet 2021, loi Climat et résilience, gel des loyers des logements F et G, interdiction de louer les logements G depuis 2025) et l'étude de référence (Chareyron, 2026).

## Partie 1. Les données

### 1.1 Sources et traitement

*À rédiger.* Les trois sources, la chaîne de traitement et les six règles d'appariement (détail en annexe A).

![Chaîne de traitement](../docs/chaine_de_traitement.png)

In [ ]:
ventes = chargement.charger_ventes()
echantillon = chargement.echantillon_analyse(ventes)
print(f"{len(ventes):,} ventes ; {ventes['apparie'].sum():,} appariées ; "
      f"{len(echantillon):,} dans l'échantillon d'analyse")

### 1.2 Le marché

Prix médian au m² de toutes les ventes, appariées ou non, par trimestre et par département.

In [ ]:
prix_trimestre = (ventes.groupby(["trimestre", "type_bien"], observed=True)["prix_m2"]
                  .median()
                  .unstack())

ax = prix_trimestre.plot(marker="o", color={"appartement": "#14213D", "maison": "#A23B3B"})
ax.set_xticks(range(len(prix_trimestre)), prix_trimestre.index, rotation=45, ha="right")
ax.set(title="Prix médian au m² par trimestre (toutes les ventes)", xlabel="", ylabel="€/m²")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
import geopandas as gpd
from matplotlib.colors import ListedColormap

# Contours des départements (projet france-geojson de Grégoire David)
URL_DEPARTEMENTS = ("https://raw.githubusercontent.com/gregoiredavid/france-geojson/"
                    "master/departements-version-simplifiee.geojson")
departements = gpd.read_file(URL_DEPARTEMENTS)

prix_dep = ventes.groupby("code_departement", observed=True)["prix_m2"].median().reset_index()
prix_dep["code_departement"] = prix_dep["code_departement"].astype(str)
carte = departements.merge(prix_dep, left_on="code", right_on="code_departement", how="left")

# Classes de prix : sinon Paris (plus de 10 000 €/m²) écraserait toutes les autres couleurs
CLASSES = [0, 1500, 2000, 2500, 3000, 4000, 20000]
NOMS = ["moins de 1 500", "1 500 à 2 000", "2 000 à 2 500", "2 500 à 3 000", "3 000 à 4 000", "plus de 4 000"]
carte["classe_prix"] = pd.cut(carte["prix_m2"], CLASSES, labels=NOMS)

bleus = ListedColormap(plt.cm.Blues(np.linspace(0.25, 1, len(NOMS))))
fig, ax = plt.subplots(figsize=(7, 7))
carte.plot(column="classe_prix", cmap=bleus, categorical=True, legend=True, ax=ax,
           edgecolor="white", linewidth=0.3,
           missing_kwds={"color": "lightgrey", "label": "hors champ"},
           legend_kwds={"title": "€/m²", "loc": "center left", "bbox_to_anchor": (1, 0.5), "frameon": False})
ax.set_title("Prix médian au m² par département, 2022 à 2025")
ax.set_axis_off()
plt.show()

En gris : la Moselle, le Bas-Rhin et le Haut-Rhin, absents de la base des ventes (régime du livre foncier).

*À rédiger.* Commentaire des deux graphiques.

### 1.3 L'échantillon

*À faire.* Un paragraphe sur le taux d'appariement (détail en annexe B), puis un tableau qui compare les ventes appariées et non appariées : prix médian au m², surface médiane, part de maisons, par année.

## Partie 2. Les résultats

Toutes les analyses de cette partie portent sur l'échantillon d'analyse : les ventes appariées, sans les ventes rapprochées.

### 2.1 Description par étiquette

*À faire.* Tableau par étiquette (prix, surface, type de bien, période de construction), carte de la part de logements F ou G par département, matrice de corrélation de Spearman.

### 2.2 Comparaison sans correction

*À faire.* Boîtes à moustaches du prix au m² par étiquette, maisons et appartements séparés.

### 2.3 Résultat principal : à logement comparable

*À faire.* Écart de prix par étiquette dans des groupes de ventes semblables (commune, type, tranche de surface, période de construction, trimestre), par rapport à G et à D, avec intervalles de confiance. Diagramme en barres : écart sans correction et écart à logement comparable.

### 2.4 Évolution de 2022 à 2025

*À faire.* Le même calcul année par année ; courbe de l'écart des classes F et G avec les dates du gel des loyers et de l'interdiction de louer.

## Partie 3. Vérification et limites

### 3.1 Appariement strict

*À faire.* Le résultat de 2.3 refait sans les ventes ambiguës et sans les ventes appariées par la distance (`chargement.appariement_strict`).

### 3.2 Limites

*À rédiger.* Reprendre les limites de la note de projet.

## Conclusion

*À rédiger.*

## Annexes

### Annexe A. Règles d'appariement

Une vente est rattachée à un DPE si les six règles suivantes sont respectées. Les seuils sont définis dans `src/config.py`.

| Règle | Contenu | Seuil |
|---|---|---|
| 1. Même adresse et même type de logement | Même commune, même voie et même numéro, après avoir écrit les adresses des deux bases de la même façon | |
| 2. À défaut, un DPE tout proche | Quand l'adresse ne permet pas de conclure, DPE situé à faible distance de la vente | 20 m pour un appartement, 60 m pour une maison |
| 3. Une surface proche | Écart entre la surface de la vente (DVF) et la surface habitable du DPE | au plus 10 % pour un appartement, 20 % pour une maison |
| 4. Un DPE établi avant la vente | DPE antérieur à la vente ; son ancienneté entre dans le score, si bien qu'à surface égale le plus récent est préféré | au plus 3 ans avant la vente |
| 5. Un DPE pour un seul logement | Un même DPE ne peut pas être attribué à deux logements différents ; il revient à celui qui lui correspond le mieux. Plusieurs ventes d'un même logement peuvent en revanche partager son DPE | |
| 6. Les cas ambigus sont signalés | Vente marquée « ambiguë » quand un autre DPE d'étiquette différente convient presque aussi bien | écart de score inférieur à 0,10 |

Quand plusieurs DPE respectent ces règles, on retient celui dont le score est le plus faible :

score = écart de surface / tolérance + 0,25 × ancienneté du DPE / 3 ans + distance / rayon (la distance ne compte que pour la règle 2).

Sont aussi écartés : les DPE dont l'adresse a été mal reconnue par la Base Adresse Nationale (note de fiabilité inférieure à 0,5), les DPE remplacés par un DPE plus récent et les DPE qui ne portent pas sur une maison ou un appartement (DPE d'immeuble entier). Les ventes d'un même logement à moins de 30 jours d'écart sont signalées et exclues de l'échantillon d'analyse.

### Annexe B. Taux d'appariement par département

Le taux par année et par type de bien est donné en partie 1.3.

In [ ]:
taux_dep = (ventes.groupby("code_departement", observed=True)["apparie"]
            .mean()
            .mul(100)
            .reset_index(name="taux"))
taux_dep["code_departement"] = taux_dep["code_departement"].astype(str)
carte_taux = departements.merge(taux_dep, left_on="code", right_on="code_departement", how="left")

CLASSES_TAUX = [0, 30, 40, 50, 60, 100]
NOMS_TAUX = ["moins de 30 %", "30 à 40 %", "40 à 50 %", "50 à 60 %", "plus de 60 %"]
carte_taux["classe_taux"] = pd.cut(carte_taux["taux"], CLASSES_TAUX, labels=NOMS_TAUX)

bleus_taux = ListedColormap(plt.cm.Blues(np.linspace(0.25, 1, len(NOMS_TAUX))))
fig, ax = plt.subplots(figsize=(7, 7))
carte_taux.plot(column="classe_taux", cmap=bleus_taux, categorical=True, legend=True, ax=ax,
                edgecolor="white", linewidth=0.3,
                missing_kwds={"color": "lightgrey", "label": "hors champ"},
                legend_kwds={"title": "Ventes appariées", "loc": "center left",
                             "bbox_to_anchor": (1, 0.5), "frameon": False})
ax.set_title("Part des ventes appariées à un DPE, 2022 à 2025")
ax.set_axis_off()
plt.show()

In [ ]:
taux_dep_annee = (ventes.groupby(["code_departement", "annee"], observed=True)["apparie"]
                  .mean()
                  .mul(100)
                  .unstack()
                  .round(1))
taux_dep_annee["ensemble"] = ventes.groupby("code_departement", observed=True)["apparie"].mean().mul(100).round(1)
taux_dep_annee.sort_values("ensemble")

### Annexe C. Ventes appariées et non appariées

Comparaison sur les variables connues pour toutes les ventes, par type de bien, sur l'ensemble de la période.

In [ ]:
statut = ventes["apparie"].map({True: "appariées", False: "non appariées"})

comparaison_detail = (ventes.groupby(["type_bien", statut], observed=True)
                      .agg(ventes=("prix_m2", "size"),
                           prix_m2_q1=("prix_m2", lambda x: x.quantile(0.25)),
                           prix_m2_median=("prix_m2", "median"),
                           prix_m2_q3=("prix_m2", lambda x: x.quantile(0.75)),
                           surface_mediane=("surface", "median"),
                           pieces_mediane=("nb_pieces", "median"))
                      .round(1))
comparaison_detail.index.names = ["type de bien", "statut"]
comparaison_detail

Répartition des étiquettes dans l'échantillon d'analyse, comparée à celle des résidences principales au 1er janvier 2024 (SDES, 2024). Le parc ne tient pas compte du changement des seuils pour les logements de moins de 40 m² entré en vigueur le 1er juillet 2024.

In [ ]:
# Résidences principales au 1er janvier 2024 (SDES, 2024), en %
PARC_2024 = pd.Series({"A": 2.6, "B": 4.6, "C": 25.7, "D": 32.1, "E": 21.1, "F": 8.1, "G": 5.7})

repartition = pd.DataFrame({
    "échantillon": echantillon["etiquette_dpe"].value_counts(normalize=True).reindex(ORDRE) * 100,
    "parc 2024": PARC_2024,
}).round(1)
display(repartition)

x = np.arange(len(ORDRE))
largeur = 0.4
fig, ax = plt.subplots()
ax.bar(x - largeur / 2, repartition["échantillon"], largeur, color=[COULEURS[e] for e in ORDRE], label="ventes appariées")
ax.bar(x + largeur / 2, repartition["parc 2024"], largeur, color="lightgrey", label="résidences principales (2024)")
ax.set_xticks(x, ORDRE)
ax.set(ylabel="%", title="Répartition des étiquettes : ventes appariées et parc de logements")
ax.legend(frameon=False)
plt.show()

### Annexe D. Dictionnaire des variables

*À rédiger.*

## Références

*À compléter à partir de la note de projet.*